# DH_Cascade_Framework_v2 — Data Layer Validation

This notebook:
1. Loads all InputData via `getDHData()`
2. Runs the full validation suite (`validate_input_data.py`)
3. Prints a summary of loaded data shapes and annual totals
4. Plots heat demand and temperature sanity-check figures

**Note:** All data is SYNTHETIC placeholder data generated by `generate_synthetic_data.py`.  
Real Estonian data must replace all files before any optimisation or publication.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

plt.rcParams.update({'figure.dpi': 130, 'font.family': 'Arial', 'font.size': 10})
print('Libraries loaded.')

## 1. Load Data via getDHData()

In [ ]:
from getDHData import getDHData, print_data_summary

print('Loading data...')
data = getDHData()
print_data_summary(data)

## 2. Run Validation Suite

In [ ]:
from validate_input_data import validate_all

results = validate_all(verbose=True)
n_pass = sum(1 for r in results.values() if r.passed)
n_fail = len(results) - n_pass
print(f'Summary: {n_pass} passed, {n_fail} failed')

## 3. Annual Heat Demand Summary

In [ ]:
print('Annual heat demand by county and cascade level (GWh/year)\n')

COUNTIES = data['Heat demand HTDHN, operationRateFix'].columns.tolist()

summary = pd.DataFrame({
    'HTDHN (GWh)':  data['Heat demand HTDHN, operationRateFix'].sum().round(1),
    'LTDHN (GWh)':  data['Heat demand LTDHN, operationRateFix'].sum().round(1),
    'VLTDHN (GWh)': data['Heat demand VLTDHN, operationRateFix'].sum().round(1),
})
summary['Total (GWh)'] = summary.sum(axis=1)
summary.loc['TOTAL'] = summary.sum()
print(summary.to_string())

## 4. Figure: Annual heat demand by county

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))

x     = np.arange(len(COUNTIES))
width = 0.28

htdhn  = data['Heat demand HTDHN, operationRateFix'].sum()
ltdhn  = data['Heat demand LTDHN, operationRateFix'].sum()
vltdhn = data['Heat demand VLTDHN, operationRateFix'].sum()

ax.bar(x - width, htdhn,  width, label='HTDHN (80–120°C)', color='#C0392B', alpha=0.85)
ax.bar(x,         ltdhn,  width, label='LTDHN (45–65°C)',  color='#E67E22', alpha=0.85)
ax.bar(x + width, vltdhn, width, label='VLTDHN (15–25°C)', color='#3498DB', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(COUNTIES, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Annual heat demand (GWh/year)')
ax.set_title('Synthetic heat demand by county and cascade level\n(PLACEHOLDER DATA — not from real Estonian measurements)', fontsize=10)
ax.legend(framealpha=0.9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../04_Results/figures/fig_heat_demand_by_county.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 04_Results/figures/fig_heat_demand_by_county.png')

## 5. Figure: Hourly heat demand time series (Harju, top 5 weeks)

In [ ]:
import datetime

hours = pd.date_range('2020-01-01', periods=8760, freq='h')

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

pairs = [
    ('Heat demand HTDHN, operationRateFix',  'HTDHN', '#C0392B'),
    ('Heat demand LTDHN, operationRateFix',  'LTDHN', '#E67E22'),
    ('Heat demand VLTDHN, operationRateFix', 'VLTDHN','#3498DB'),
]
for ax, (key, label, col) in zip(axes, pairs):
    ts = data[key]['Harju']
    ax.fill_between(hours, ts, alpha=0.5, color=col)
    ax.plot(hours, ts, color=col, linewidth=0.4)
    ax.set_ylabel(f'{label}\n(GWh/h)', fontsize=9)
    ax.grid(alpha=0.3)

axes[0].set_title('Harju county — synthetic hourly heat demand (2020 reference year)\n'
                  '[PLACEHOLDER DATA]', fontsize=10)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
plt.tight_layout()
plt.savefig('../04_Results/figures/fig_heat_demand_timeseries_Harju.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 04_Results/figures/fig_heat_demand_timeseries_Harju.png')

## 6. Figure: Temperature profiles — Harju county

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Upper: ambient and network supply temperatures (°C)
ax = axes[0]
T_amb  = data['ERA5 ambient temperature, temperatureTimeSeries']['Harju'] - 273.15
T_sup_H = data['HTDHN network, supplyTemperature']['Harju'] - 273.15
T_ret_H = data['HTDHN network, returnTemperature']['Harju'] - 273.15
T_sup_L = data['LTDHN network, supplyTemperature']['Harju'] - 273.15
T_ret_L = data['LTDHN network, returnTemperature']['Harju'] - 273.15

ax.plot(hours, T_amb,   color='grey',    lw=0.6, label='Ambient (ERA5)', alpha=0.7)
ax.plot(hours, T_sup_H, color='#C0392B', lw=0.8, label='HTDHN supply')
ax.plot(hours, T_ret_H, color='#E74C3C', lw=0.6, label='HTDHN return', linestyle='--')
ax.plot(hours, T_sup_L, color='#E67E22', lw=0.8, label='LTDHN supply')
ax.plot(hours, T_ret_L, color='#F39C12', lw=0.6, label='LTDHN return', linestyle='--')
ax.set_ylabel('Temperature (°C)')
ax.legend(ncol=3, fontsize=8, framealpha=0.9)
ax.grid(alpha=0.3)
ax.set_title('Harju county — network temperature profiles [PLACEHOLDER DATA]', fontsize=10)

# Lower: VLTDHN and Carnot factor preview
ax2 = axes[1]
T_sup_V = data['VLTDHN network, supplyTemperature']['Harju'] - 273.15
T_ret_V = data['VLTDHN network, returnTemperature']['Harju'] - 273.15

# Carnot factor for HTDHN (preview of exergy layer calculation)
T0  = data['ERA5 ambient temperature, temperatureTimeSeries']['Harju'].values
Ts_H = data['HTDHN network, supplyTemperature']['Harju'].values
Tr_H = data['HTDHN network, returnTemperature']['Harju'].values
T_lm_H = (Ts_H - Tr_H) / np.log(np.maximum(Ts_H / Tr_H, 1.001))
carnot_H = 1 - T0 / T_lm_H

Ts_L = data['LTDHN network, supplyTemperature']['Harju'].values
Tr_L = data['LTDHN network, returnTemperature']['Harju'].values
T_lm_L = (Ts_L - Tr_L) / np.log(np.maximum(Ts_L / Tr_L, 1.001))
carnot_L = 1 - T0 / T_lm_L

ax2.plot(hours, T_sup_V, color='#3498DB', lw=0.8, label='VLTDHN supply')
ax2.plot(hours, T_ret_V, color='#85C1E9', lw=0.6, label='VLTDHN return', linestyle='--')
ax2.set_ylabel('Temperature (°C)')
ax2_r = ax2.twinx()
ax2_r.plot(hours, carnot_H, color='#C0392B', lw=0.5, alpha=0.6, label='Carnot ψ HTDHN')
ax2_r.plot(hours, carnot_L, color='#E67E22', lw=0.5, alpha=0.6, label='Carnot ψ LTDHN')
ax2_r.set_ylabel('Carnot factor ψ (—)', color='#7D3C98')
ax2_r.set_ylim(0, 0.35)

lines1, lab1 = ax2.get_legend_handles_labels()
lines2, lab2 = ax2_r.get_legend_handles_labels()
ax2.legend(lines1+lines2, lab1+lab2, ncol=4, fontsize=8, framealpha=0.9)
ax2.grid(alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
plt.tight_layout()
plt.savefig('../04_Results/figures/fig_temperature_profiles_Harju.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 04_Results/figures/fig_temperature_profiles_Harju.png')

## 7. Figure: Heat source potentials by county

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))

x     = np.arange(len(COUNTIES))
width = 0.28

iwh    = data['Industrial waste heat, capacityMax'] * 8760 / 1.5  # back to GWh/yr for display
geo    = data['Geothermal heat, capacityMax'] * 8760 / 1.2
biomass= data['Biomass boiler, capacityMax'] * 8760 / 4.0 * 0.25  # at 25% CF

ax.bar(x - width, iwh,     width, label='Industrial waste heat', color='#2C3E50', alpha=0.85)
ax.bar(x,         geo,     width, label='Geothermal',            color='#27AE60', alpha=0.85)
ax.bar(x + width, biomass, width, label='Biomass (at 25% CF)',   color='#8B4513', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(COUNTIES, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Annual potential (GWh/year)')
ax.set_title('Heat source potentials by county\n[SYNTHETIC PLACEHOLDER DATA]', fontsize=10)
ax.legend(framealpha=0.9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../04_Results/figures/fig_heat_source_potentials.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Figure: DH pipe network eligibility

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

elig = data['DH pipes, eligibility']
dist = data['DH pipes, distances']

# Eligibility matrix
im1 = axes[0].imshow(elig.values, cmap='Blues', vmin=0, vmax=1, aspect='auto')
axes[0].set_xticks(range(len(COUNTIES)))
axes[0].set_yticks(range(len(COUNTIES)))
axes[0].set_xticklabels(COUNTIES, rotation=90, fontsize=7)
axes[0].set_yticklabels(COUNTIES, fontsize=7)
axes[0].set_title(f'DH Pipe Eligibility\n({int(elig.values.sum())//2} eligible connections)')
plt.colorbar(im1, ax=axes[0], shrink=0.8)

# Distance matrix (eligible routes only)
dist_masked = dist.values.copy()
dist_masked[dist_masked == 0] = np.nan
im2 = axes[1].imshow(dist_masked, cmap='YlOrRd', aspect='auto')
axes[1].set_xticks(range(len(COUNTIES)))
axes[1].set_yticks(range(len(COUNTIES)))
axes[1].set_xticklabels(COUNTIES, rotation=90, fontsize=7)
axes[1].set_yticklabels(COUNTIES, fontsize=7)
axes[1].set_title('Eligible Pipe Distances (km)\n(NaN = not eligible)')
plt.colorbar(im2, ax=axes[1], shrink=0.8, label='km')

plt.suptitle('DH Network Structure [SYNTHETIC DATA]', y=1.01)
plt.tight_layout()
plt.savefig('../04_Results/figures/fig_dh_network_structure.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Cost Assumptions Summary

In [ ]:
print('\n=== Heat Technology Cost Assumptions ===')
print(data['Heat technologies, costAssumptions'][[
    'investPerCapacity_BnEuro_per_GW',
    'opexPerCapacity_BnEuro_per_GW_yr',
    'lifetime_yr',
    'eta_or_COP_ref'
]].to_string())

print('\n=== Thermal Storage Cost Assumptions ===')
print(data['Thermal storage, costAssumptions'][[
    'investPerCapacity_BnEuro_per_GWh',
    'chargeEff', 'selfDischarge_per_hour', 'lifetime_yr'
]].to_string())

print('\n=== DH Pipe Cost Assumptions ===')
print(data['DH pipes, costAssumptions'].to_string())

## 10. v2 Status Summary

In [ ]:
print('=' * 60)
print('DH_Cascade_Framework_v2 — Data Layer Status')
print('=' * 60)

n_pass = sum(1 for r in results.values() if r.passed)
print(f'Validation:   {n_pass}/{len(results)} checks passed')
print(f'Data keys:    {len(data)}')
print(f'Counties:     15 (Estonian maakond)')
print(f'Timesteps:    8760 h (2020 reference year)')
print()
print('Annual heat demand totals:')
for level in ('HTDHN', 'LTDHN', 'VLTDHN'):
    key = f'Heat demand {level}, operationRateFix'
    total = data[key].values.sum()
    print(f'  {level}: {total:.0f} GWh/year')
print()
print('DATA STATUS: ALL SYNTHETIC — replace before publication')
print()
print('Real-data gaps remaining:')
gaps = [
    '[G1] ERA5 temperature → download from CDS Copernicus',
    '[G2] Heat demand      → Estonian Heat Atlas / Statistikaamet',
    '[G3] Industrial WH    → real process temperature logs',
    '[G4] Geothermal       → TUT geothermal survey',
    '[G5] Biomass          → ENMAK 2030 county breakdown',
    '[G6] Solar thermal CF → ERA5 GHI / PVGIS API',
    '[G7] Network geometry → Elveso GIS data',
    '[G8] Cost data        → IEA DH reports / Estonian DH operators',
]
for g in gaps:
    print(f'  {g}')
print()
print('Ready for v3:')
print('  buildModel.py — EnergySystemModel + components')
print('  runOptimisation.py — TSA + Gurobi solve')
print('  Estonia_2025_BaselineA.ipynb — first case study')
print('=' * 60)